# Faz 1+2 — Veri Hazırlık ve Split Ön İşleme
### Kaggle · Google Colab · Yerel Makine

**Bu notebook diğer tüm Faz notebook'larından önce çalıştırılmalıdır.**

Çıktılar:
| Dosya | Yol | Kullanan |
|-------|-----|---------|
| `manifest.csv` | `outputs/splits/` | nnUNet, SwinTransformer, MedNeXt |
| `splits.csv`, `fold0_train.csv`, `fold0_val.csv` | `outputs/splits/` | Tüm modeller |
| `holdout.csv` | `outputs/splits/` | Harici test |
| `det_data/fold0/` (PNG + YOLO labels) | `outputs/det_data/` | YOLO |

**Kaggle workflow:**
1. Bu notebook'u çalıştır → çıktıları `/kaggle/working/` altına yazar
2. `Save Version` → çıktıları yeni dataset olarak kaydet
3. Diğer Faz notebook'larında bu dataset'i input olarak ekle


---
## 0. Ortam Tespiti

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f"Ortam: {env_name}")
import platform; print(f"Python: {sys.version.split()[0]}  |  OS: {platform.system()}")


---
## 1. Ortam Kurulumu (Colab için)

In [ ]:
if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata, drive
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        drive.mount('/content/drive', force_remount=False)
        print("Kaggle kimlik bilgileri + Google Drive bağlandı")
    except Exception as e:
        print(f"Manuel kurulum gerekebilir: {e}")
else:
    print(f"{env_name} — Colab kurulumu atlandı")


---
## 2. Kütüphane Kurulumu

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'pandas', 'numpy', 'scikit-learn', 'openpyxl',
     'pydicom', 'Pillow', 'tqdm'],
    check=True
)
import importlib; importlib.invalidate_caches()
import pandas as pd, numpy as np, pydicom
from PIL import Image
from tqdm.notebook import tqdm
print(f"pandas   : {pd.__version__}")
print(f"pydicom  : {pydicom.__version__}")


---
## 3. Konfigürasyon

**Kaggle:** `Bilgi.xlsx` ve DICOM klasörleri `/kaggle/input/{KAGGLE_DATASET_SLUG}/` altında olmalıdır.  
**Colab:** Dataset indirilir; aynı yapıya beklenir.


In [ ]:
# ─── Kullanıcı Ayarları ───────────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen-acikveri'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'
GITHUB_URL          = 'https://github.com/ramazan2020/abdomen1.git'
N_FOLDS             = 5   # GroupKFold fold sayısı
SEED                = 42
HOLDOUT_FRAC        = 0.15
EXPORT_YOLO_FOLD    = 0   # Hangi fold için YOLO det_data oluşturulsun
# ─────────────────────────────────────────────────────────────────────────

if IS_KAGGLE:
    DATA_DIR  = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR  = Path('/kaggle/working')

elif IS_COLAB:
    DATA_DIR  = Path('/content/kaggle_data')
    DRIVE_BASE= Path('/content/drive/MyDrive/Abdomen')
    WORK_DIR  = Path('/content/abdomen_work')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    PROJECT_ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    DATA_DIR  = PROJECT_ROOT
    WORK_DIR  = PROJECT_ROOT / 'outputs'

# Ortak yollar
EGITIM_DIR  = DATA_DIR / 'Egitim Verisi'
YARISMA_DIR = DATA_DIR / 'Yarisma Verisi'
BILGI_XLSX  = DATA_DIR / 'Bilgi.xlsx'
SPLIT_DIR   = WORK_DIR / 'splits'
DET_DATA_DIR= WORK_DIR / 'det_data'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
DET_DATA_DIR.mkdir(parents=True, exist_ok=True)

# src/config.py import ÖNCE env var'ları ayarla
os.environ['ABDOMEN_BILGI_XLSX']    = str(BILGI_XLSX)
os.environ['ABDOMEN_TRAIN_DIR']     = str(EGITIM_DIR)
os.environ['ABDOMEN_TEST_DIR']      = str(YARISMA_DIR)
os.environ['ABDOMEN_SPLIT_DIR']     = str(SPLIT_DIR)
os.environ['ABDOMEN_DET_DATA_DIR']  = str(DET_DATA_DIR)
os.environ['ABDOMEN_OUT_DIR']       = str(WORK_DIR)

print(f"DATA_DIR    : {DATA_DIR}")
print(f"WORK_DIR    : {WORK_DIR}")
print(f"EGITIM_DIR  : {EGITIM_DIR}  ({'✓' if EGITIM_DIR.exists() else '✗'})")
print(f"BILGI_XLSX  : {BILGI_XLSX}  ({'✓' if BILGI_XLSX.exists() else '✗ — henüz indirilmedi'})")
print(f"SPLIT_DIR   : {SPLIT_DIR}")
print(f"DET_DATA_DIR: {DET_DATA_DIR}")


In [ ]:
# ── GitHub Repo / Local src ───────────────────────────────────────────────
if IS_LOCAL:
    REPO_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    print(f'Local: src/ → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(['git','clone','--depth=1', GITHUB_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.config        import SUPER_CLASSES, RAW_PATHOLOGY_TO_SUPER, DEFAULT_SPLIT
from src.splits        import load_merged_annotations, make_splits, load_fold, load_holdout
from src.preprocessing import build_manifest

print('src modülleri ✓')
print(f'6 üst sınıf: {SUPER_CLASSES}')


---
## 4. Veri İndirme (Yalnızca Colab)

Kaggle'da bu adım atlanır — dataset zaten mount edilmiş olur.

In [ ]:
if IS_KAGGLE:
    print("Kaggle: Dataset mount edilmiş")
    _egitim_cnt = len(list(EGITIM_DIR.iterdir())) if EGITIM_DIR.exists() else 0
    print(f"  Egitim Verisi : {_egitim_cnt} vaka klasörü")
    print(f"  Bilgi.xlsx    : {'✓' if BILGI_XLSX.exists() else '✗'}")
else:
    _already = BILGI_XLSX.exists() and EGITIM_DIR.exists()
    if _already:
        print(f"Veri mevcut: {DATA_DIR}")
    elif not IS_LOCAL:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Dataset indiriliyor: {KAGGLE_DATASET_ID}")
        print("Bu işlem 30-90 dakika sürebilir...")
        t0 = time.time()
        r = subprocess.run(
            ['kaggle', 'datasets', 'download',
             '-d', KAGGLE_DATASET_ID, '-p', str(DATA_DIR), '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('HATA:', r.stderr[-800:])
            raise RuntimeError('İndirme başarısız')
        print(f"İndirildi ({time.time()-t0:.0f}s)")

assert BILGI_XLSX.exists(), f'Bilgi.xlsx bulunamadı: {BILGI_XLSX}'
assert EGITIM_DIR.exists(), f'Egitim Verisi bulunamadı: {EGITIM_DIR}'
egitim_cases = [d for d in EGITIM_DIR.iterdir() if d.is_dir()]
print(f'\nEğitim vaka sayısı : {len(egitim_cases)}')
print(f'İlk 5: {[d.name for d in sorted(egitim_cases)[:5]]}')


---
## 5. Bilgi.xlsx Keşfi

In [ ]:
xl = pd.ExcelFile(BILGI_XLSX)
print(f"Sayfalar: {xl.sheet_names}")

train_df = pd.read_excel(xl, sheet_name='TRAIININGDATA')
comp_df  = pd.read_excel(xl, sheet_name='COMPETITIONDATA')

print(f"\nTRAIININGDATA   : {len(train_df):,} annotasyon, {train_df['Case Number'].nunique()} vaka")
print(f"COMPETITIONDATA : {len(comp_df):,} annotasyon, {comp_df['Case Number'].nunique()} vaka")
print(f"Sütunlar: {list(train_df.columns)}")

print("\nType dağılımı (TRAININGDATA):")
print(train_df['Type'].value_counts().to_string())

print("\nClass dağılımı (TRAININGDATA, ilk 15):")
print(train_df['Class'].value_counts().head(15).to_string())


---
## 6. Ham Etiket → 6 Üst Sınıf Eşleme Kontrolü

In [ ]:
full_df = pd.concat([train_df.assign(prefix='T'), comp_df.assign(prefix='C')], ignore_index=True)
all_classes = full_df['Class'].value_counts()

mapped    = {c: n for c, n in all_classes.items() if c in RAW_PATHOLOGY_TO_SUPER}
unmapped  = {c: n for c, n in all_classes.items() if c not in RAW_PATHOLOGY_TO_SUPER}

print(f"Eşlenen patoloji sınıfları : {len(mapped)}")
print(f"Eşlenmeyen / anatomik      : {len(unmapped)}")
print()
print(f"{'Ham Sınıf':<50} {'Üst Sınıf ID':<5} {'Üst Sınıf Adı'}")
print('─' * 85)
for raw, sid in sorted(RAW_PATHOLOGY_TO_SUPER.items(), key=lambda x: x[1]):
    cnt = all_classes.get(raw, 0)
    print(f"  {raw:<48} {sid:<5} {SUPER_CLASSES[sid]}  ({cnt:,} annot)")


---
## 7. Manifest Oluşturma

Her `(vaka, kesit)` için etiket ve bbox bilgilerini içeren `manifest.csv` üretilir.
Bu dosya nnUNet, SwinTransformer ve MedNeXt notebook'larının girdisidir.

⏱ **Süre:** DICOM'ları okumadan yalnızca xlsx'i işler → 1-3 dakika.


In [ ]:
manifest_path = SPLIT_DIR / 'manifest.csv'

if manifest_path.exists():
    print(f'Manifest zaten mevcut: {manifest_path}')
    print('Yeniden oluşturmak için dosyayı silin.')
    manifest = pd.read_csv(manifest_path)
else:
    print('Manifest oluşturuluyor...')
    t0 = time.time()
    manifest_path = build_manifest(manifest_path)
    manifest = pd.read_csv(manifest_path)
    print(f'Tamamlandı ({time.time()-t0:.0f}s)')

print(f'\nManifest: {len(manifest):,} satır, {manifest["case"].nunique()} tekil vaka')
print(f'Sütunlar: {list(manifest.columns)}')
print()
manifest['has_bbox'] = manifest['bboxes'].fillna('').astype(str).str.len() > 2
print(f'BBox annotasyonlu kesit: {manifest["has_bbox"].sum():,}')
print(f'Kaynak dağılımı:')
print(manifest['source'].value_counts().to_string())


---
## 8. GroupKFold Split Oluşturma

Aynı vakanın kesitleri ASLA birden fazla fold'a düşmez (data leak yok).
- `K = 5` fold
- Hold-out = %15 → harici test için ayrılır


In [ ]:
splits_path = SPLIT_DIR / 'splits.csv'

if splits_path.exists():
    print(f'Split zaten mevcut: {splits_path}')
    print('Yeniden oluşturmak için splits.csv, holdout.csv ve fold*.csv dosyalarını silin.')
else:
    print('Split oluşturuluyor...')
    paths = make_splits(out_dir=SPLIT_DIR)
    print('Oluşturulan dosyalar:')
    for k, v in sorted(paths.items()):
        print(f'  {k}: {v.name}  ({v.stat().st_size} byte)')

# Özet
holdout_cases = load_holdout()
fold0_train   = load_fold(0, 'train')
fold0_val     = load_fold(0, 'val')

print(f'\nHold-out vaka : {len(holdout_cases)}')
print(f'Fold 0 train  : {len(fold0_train)}')
print(f'Fold 0 val    : {len(fold0_val)}')

# Sınıf dağılımı kontrolü
manifest2 = pd.read_csv(SPLIT_DIR / 'manifest.csv')
manifest2['super_labels'] = manifest2['super_labels'].fillna('').astype(str)

def count_cls(cases):
    sub = manifest2[manifest2['case'].isin(cases)]
    cnt = {s: 0 for s in range(6)}
    for v in sub['super_labels']:
        for s in (v.split(';') if v else []):
            if s.strip(): cnt[int(s)] += 1
    return cnt

rows = []
for name, cs in [('train', fold0_train), ('val', fold0_val), ('holdout', holdout_cases)]:
    cnt = count_cls(cs)
    rows.append([name, len(cs)] + [cnt[i] for i in range(6)])
dist = pd.DataFrame(rows, columns=['split', 'n_case'] + SUPER_CLASSES)
print()
print(dist.to_string(index=False))


---
## 9. YOLO Det Data Oluşturma

Her kesiti PNG'ye dönüştürür ve YOLO format etiket dosyası (`.txt`) üretir.

⏱ **Süre:** DICOM okuma + PNG yazma — **30-90 dakika** (GPU gerekmez).  
⚠ **Boyut:** ~5-15 GB disk alanı gerekebilir.


In [ ]:
from src.detection import export_yolo_dataset

fold_dir = DET_DATA_DIR / f'fold{EXPORT_YOLO_FOLD}'
_train_imgs = list((fold_dir / 'images' / 'train').glob('*.png')) if fold_dir.exists() else []
_val_imgs   = list((fold_dir / 'images' / 'val').glob('*.png'))   if fold_dir.exists() else []

if _train_imgs:
    print(f'Det data zaten mevcut: {fold_dir}')
    print(f'  train PNG: {len(_train_imgs):,}')
    print(f'  val   PNG: {len(_val_imgs):,}')
    print('Yeniden oluşturmak için fold_dir/images/ klasörünü silin.')
else:
    print(f'YOLO det data oluşturuluyor (fold={EXPORT_YOLO_FOLD})...')
    print(f'Kaynak: {EGITIM_DIR}')
    print(f'Hedef : {fold_dir}')
    print('Bu işlem 30-90 dakika sürebilir...')
    t0 = time.time()
    result_dir = export_yolo_dataset(fold=EXPORT_YOLO_FOLD, include_val_negatives=False, bbox_only=True)
    elapsed = time.time() - t0
    print(f'\nTamamlandı ({elapsed/60:.0f} dakika)')

    _train_imgs = list((result_dir / 'images' / 'train').glob('*.png'))
    _val_imgs   = list((result_dir / 'images' / 'val').glob('*.png'))
    print(f'  train PNG: {len(_train_imgs):,}')
    print(f'  val   PNG: {len(_val_imgs):,}')

# dataset.yaml göster
_yaml = fold_dir / 'dataset.yaml'
if _yaml.exists():
    print(f'\ndataset.yaml ({_yaml}):')
    print(_yaml.read_text())


---
## 10. Çıktı Özeti ve Boyut Kontrolü

In [ ]:
import shutil

print('=== Çıktı Özeti ===')
print(f'SPLIT_DIR    : {SPLIT_DIR}')
for f in sorted(SPLIT_DIR.glob('*.csv')):
    print(f'  {f.name:<30} {f.stat().st_size/1e3:>8.0f} KB')

print()
det_fold = DET_DATA_DIR / f'fold{EXPORT_YOLO_FOLD}'
if det_fold.exists():
    _size_mb = sum(f.stat().st_size for f in det_fold.rglob('*') if f.is_file()) / 1e6
    _train_n = len(list((det_fold / 'images' / 'train').glob('*.png'))) if (det_fold/'images'/'train').exists() else 0
    _val_n   = len(list((det_fold / 'images' / 'val').glob('*.png')))   if (det_fold/'images'/'val').exists()   else 0
    print(f'DET_DATA_DIR : {det_fold}')
    print(f'  train PNG  : {_train_n:,}')
    print(f'  val   PNG  : {_val_n:,}')
    print(f'  Toplam boyut: {_size_mb:.0f} MB')

print()
print('=== Sonraki Adımlar ===')
print('Kaggle: Save Version → çıktıları yeni dataset olarak ekle')
print()
print('Notebook sırası:')
print('  ✅ Faz1_2_VeriHazirlik_Colab_Kaggle  ← bu notebook')
print('  →  Faz3_YOLO_Colab_Kaggle')
print('  →  Faz3_nnUNet_Colab_Kaggle')
print('  →  Faz4_SwinTransformer_Colab_Kaggle')
print('  →  Faz5_MedNeXt_Colab_Kaggle')
